# Jupyter и научный Python: NumPy и Matplotlib

## Мотивация

Почти вся исследовательская работа в ML начинается в **ноутбуке**: там удобно по шагам проверять идеи и сразу видеть числа и графики рядом с кодом. Тяжёлые вычисления при этом доверяют не циклам Python, а векторизованному **NumPy**. Итог почти всегда нужно показать глазами — это делает **Matplotlib**.

Сегодня соберём этот ежедневный набор: интерактивный ноутбук и его магии → быстрые массивы → графики. На нём держатся все дальнейшие семинары.

> Код сегодня выполняется в **уже готовом окружении** — на учебном сервере или в Colab. Как окружение собирают самому (`python -m venv`, `uv`, файлы зависимостей) — семинар 5.

## 1. Jupyter Notebook: как он устроен

Ноутбук — это последовательность **ячеек**: с кодом или с текстом (Markdown). Код исполняет **ядро** (kernel) — отдельный процесс Python, который хранит состояние между ячейками: переменная, созданная в одной ячейке, видна в следующих. Число `[N]` слева от ячейки — порядковый номер её запуска, а не позиция в ноутбуке.

- `Shift+Enter` — выполнить ячейку и перейти к следующей.
- `Esc` — командный режим, и в нём: `B` / `A` — добавить ячейку ниже/выше, `M` / `Y` — сделать её markdown-ячейкой или снова кодовой, `DD` — удалить.
- `Tab` — автодополнение, `Shift+Tab` — подсказка по сигнатуре.
- Kernel → Restart & Run All — перезапустить ядро и выполнить всё сверху вниз.

Значение **последнего выражения** ячейки печатается само, без `print` — этим удобно смотреть промежуточный результат.

### Проверка воспроизводимости

В конце работы перезапустите ядро и выполните все ячейки сверху вниз (Kernel → Restart & Run All) — и убедитесь, что ноутбук отрабатывает **без ошибок**. Так проверяется воспроизводимость: итог не должен зависеть от того, в каком порядке вы запускали ячейки во время работы.

### Ноутбук или `.py`-модуль

Ноутбук удобен для обучения и разведки: запустил маленький кусочек кода — сразу увидел результат. Но когда кода становится много, поддерживать его в ноутбуке практически невозможно — теряются структура, переиспользование и тестируемость. Поэтому «боевой» код принято держать в репозитории: модулями `.py` под контролем версий (а всё чаще — редактируя их кодовыми агентами). Практическое правило: прототип и разведка — в ноутбуке, а устоявшуюся логику переносите в модули репозитория.

In [ ]:
answer = 42                 # переменная останется в памяти ядра
greeting = "hello, kernel"
answer * 2                  # значение последнего выражения печатается без print

Текст ячеек и содержимое памяти ядра — две разные вещи, и они расходятся, как только ячейки запускают не по порядку или правят, не запуская:

Ноутбук — это текст, состояние живёт в памяти ядра

#### ❓ **Вопрос**: Почему один и тот же ноутбук может дать разный результат при запуске ячеек в разном порядке?

<details>

<summary><strong>Ответ</strong></summary>

Ядро хранит состояние (значения переменных) и меняет его в том порядке, в котором вы запускаете ячейки, а не в котором они расположены. Если выполнить ячейки не по порядку или несколько раз, значение переменной может разойтись с кодом выше. Надёжная проверка воспроизводимости — Restart & Run All: перезапуск ядра и выполнение всех ячеек сверху вниз.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

IPython начал в 2001 году аспирант-физик Фернандо Перес: ему не хватало интерактивной среды уровня Mathematica, и первая версия была написана за выходные и уместилась в пару сотен строк. В 2014-м проект переименовали в Jupyter — имя собрано из **Ju**lia, **Pyt**hon и **R**, трёх языков, для которых тогда были ядра, и заодно отсылает к тетрадям Галилея с наблюдениями спутников Юпитера. В 2017-м Jupyter получил ACM Software System Award — ту же премию, что когда-то Unix, TeX и WWW.

Мораль для студентов простая: ноутбук — не «облегчённый Python для новичков», а инструмент, на котором опубликованы расчёты к статьям в Nature и данные LIGO.

</details>

### Где будут лежать файлы

Всё, что ноутбук сохранит на диск (картинки, данные), должно пережить занятие, поэтому работаем в домашнем каталоге студента — `~/seminar-01/`, а не в `/tmp`: временный каталог вычищается при перезагрузке машины. Перейдём туда один раз, и дальше все относительные пути будут вести в него.

`Path.home()` — это и есть домашний каталог текущего пользователя (`/home/студент` на учебном сервере, `/root` в Colab), `mkdir` создаёт каталог, а `os.chdir` меняет **рабочий каталог ядра**: после него `savefig("plot.png")` без всякого пути положит файл именно туда.

Оговорка про Colab: там временная вся машина целиком, включая домашний каталог, — в Colab результаты сохраняют на подключённый Google Drive (раздел 6).

In [ ]:
from pathlib import Path
import os

work_dir = Path.home() / "seminar-01"   # ~/seminar-01 переживёт перезагрузку, /tmp — нет
work_dir.mkdir(exist_ok=True)           # создаём один раз; повторный запуск ячейки не мешает
os.chdir(work_dir)                      # с этого момента savefig пишет сюда
print("работаем в:", Path.cwd())        # проверяем, что ядро действительно переехало

## 2. Магии ноутбука: `%` и `%%`

**Магии** — специальные команды IPython-ядра, которых нет в обычном Python:

- **Строчная магия** `%` действует на одну строку: `%timeit`, `%pwd`, `%who`, `%run script.py`.
- **Ячейковая магия** `%%` действует на **всю ячейку** и стоит первой строкой: `%%time`, `%%bash`, `%%writefile file.py`.
- `!команда` выполняет команду shell: `!pip install numpy`, `!nvidia-smi`.

Частые магии: `%timeit` — усреднённый замер времени выражения; `%%time` — время всей ячейки; `%who` / `%whos` — список определённых переменных.

Раньше вывод графиков Matplotlib «включали» магией `%matplotlib inline`. Сейчас она почти везде не нужна — в свежих Jupyter и в Google Colab отрисовка графиков в ноутбук включена по умолчанию.

In [ ]:
!python --version           # ! — команда shell прямо из ноутбука
%pwd                        # текущий каталог ядра — тот, куда мы перешли выше
%who                        # какие переменные уже определены в ядре
%timeit sum(range(100_000)) # усреднённый замер времени выражения

Ячейковая магия — другое дело: она стоит **первой строкой** и относится ко всему, что в ячейке. `%%time` измеряет, сколько шло всё её содержимое:

In [ ]:
%%time
total = sum(i * i for i in range(1_000_000))   # обычный цикл Python — заметно небыстрый
print("сумма:", total)                         # снизу IPython допечатает время всей ячейки

А `%%writefile` ячейку вообще не выполняет — она сохраняет её текст в файл. Так прямо из ноутбука получается настоящий скрипт (он ляжет в `~/seminar-01`, куда мы перешли выше):

In [ ]:
%%writefile hello.py
# всё, что ниже строки %%writefile, уходит в файл, а не исполняется
print("привет из скрипта")

Файл записан — запустим его строчной магией `%run` и заодно убедимся, что он действительно лежит в рабочем каталоге:

In [ ]:
%run hello.py      # ядро выполняет скрипт как обычный python-файл
!ls -l hello.py    # файл на месте, в ~/seminar-01; -l показывает размер и дату

#### ❓ **Вопрос**: В ячейке первой строкой стоит `x = 5`, а второй — `%%time`. Что произойдёт при запуске?

<details>

<summary><strong>Ответ</strong></summary>

Ячейка упадёт: `UsageError: Line magic function `%%time` not found`. Ячейковую магию IPython распознаёт **только в самой первой строке** ячейки — там она относится ко всему коду ниже. Строкой ниже это уже не «магия ячейки», и IPython пытается понять `%%time` как строчную магию, которой не существует.

Строчная магия так не ограничена: `%timeit`, `%run`, `%who` относятся к своей строке и могут стоять где угодно в ячейке.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

`%timeit` — не «замерить один раз»: он сам подбирает число повторений, гоняет несколько серий, печатает среднее с разбросом и на время замера отключает сборщик мусора. Поэтому `%timeit` и `%%time` на одном и том же коде спокойно расходятся в разы — это не баг, это разные вещи: устоявшееся время против одного прогона со всеми накладными расходами.

Ловушка, на которой все спотыкаются хотя бы раз: если выражение ленивое (генератор, отложенная операция dask, вызов на GPU), таймер померяет время создания объекта, а не время работы — и вы получите «ускорение в тысячу раз», которого нет.

</details>

## 3. NumPy: массивы и векторизация

`ndarray` — массив чисел одного типа, лежащих в памяти подряд. Ключевые атрибуты — `shape` (форма) и `dtype` (тип элементов); это **атрибуты, а не методы**, пишутся без скобок. Массивы создают через `np.array`, `np.zeros`, `np.ones`, `np.arange`, `np.linspace`.

Два последних легко перепутать: `np.arange(0, 1, 0.25)` задаёт **шаг** и правую границу не включает (как обычный `range`), а `np.linspace(0, 1, 5)` задаёт **число точек** и включает обе границы. Для сетки под график почти всегда нужен `linspace` — иначе правый край молча потеряется.

Операции применяются **поэлементно, без циклов** (векторизация): `a ** 2`, `np.sqrt(a)`, `a + b`. **Broadcasting** «растягивает» массивы совместимых форм: `a + 10` прибавляет скаляр к каждому элементу. Векторизованный код короче и в десятки раз быстрее цикла Python, потому что операция выполняется единым проходом внутри скомпилированного кода над непрерывной памятью.

In [ ]:
import numpy as np

a = np.array([1, 4, 9, 16, 25])
print("shape:", a.shape, "| dtype:", a.dtype)   # атрибуты, без ()
print("zeros:", np.zeros(3), "| ones:", np.ones(3))   # заготовки нужной формы, dtype float
print("linspace:", np.linspace(0, 1, 5))        # 5 точек от 0 до 1 включительно

Операции применяются к массиву целиком, поэлементно — циклы не нужны:

In [ ]:
print("a ** 2  =", a ** 2)      # поэлементно
print("sqrt(a) =", np.sqrt(a))   # np.sqrt — корень (np.square — квадрат!)
print("a + 10  =", a + 10)       # скаляр прибавился к каждому элементу
print("список  =", [1, 4, 9] + [10])   # а для списка «+» — это склейка

#### ❓ **Вопрос**: Почему `[1, 4, 9] + [10]` и `a + 10` в ячейке выше дали разное?

<details>

<summary><strong>Ответ</strong></summary>

Для списка Python оператор `+` — это **конкатенация**: `[1, 4, 9] + [10]` даёт `[1, 4, 9, 10]`, список стал длиннее на один элемент. Для массива NumPy `+` — **поэлементная** операция, и скаляр `10` по правилам broadcasting прибавляется к каждому элементу: `[1, 4, 9, 16, 25]` превращается в `[11, 14, 19, 26, 35]`, длина не меняется.

</details>

In [ ]:
big = np.arange(1_000_000)

%timeit sum(i * i for i in range(1_000_000))    # чистый Python — медленно
%timeit (big * big).sum()                        # векторизованный NumPy — быстро

Дело не в том, что «NumPy умный», а в том, как лежат данные и кто крутит цикл:

Список указателей против непрерывного массива

#### ❓ **Вопрос**: Почему `(big * big).sum()` в разы быстрее цикла Python по тем же числам?

<details>

<summary><strong>Ответ</strong></summary>

В цикле Python на каждый элемент создаётся объект и выполняется интерпретируемый код. NumPy хранит числа одного типа подряд в памяти и выполняет операцию единым проходом внутри скомпилированного кода (C), без пооэлементных Python-объектов и накладных расходов интерпретатора.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

NumPy — результат примирения двух конкурировавших библиотек: Numeric (1995, Джим Хугунин) и numarray. В 2005–2006 Трэвис Олифант собрал из них один пакет, чтобы сообщество перестало писать каждый расчёт в двух вариантах. Ставка сыграла: сегодня SciPy, pandas, scikit-learn и PyTorch говорят про массивы на языке NumPy, а `ndarray` де-факто стал общим языком научного Python.

В 2020-м про NumPy вышла статья в Nature — редкий случай, когда в такой журнал попадает библиотека, а не результат. В ней среди прочего вспоминают, что на NumPy считали данные LIGO и первое изображение чёрной дыры.

</details>

### Индексирование и срезы

Как у списков: `x[2]` — элемент, `x[2:5]` — полуоткрытый интервал (с 2-го по 4-й включительно, правая граница не входит), `x[-3:]` — три последних. Но у массивов есть то, чего у списков нет — **булева маска**: сравнение даёт массив из `True`/`False`, и им можно индексировать, оставляя только подходящие элементы.

In [ ]:
x = np.arange(10)

print("x[2:5]  =", x[2:5])          # с 2-го по 4-й
print("x[-3:]  =", x[-3:])          # три последних
print("маска   =", x % 2 == 0)      # массив True/False
print("чётные  =", x[x % 2 == 0])   # индексируем маской

### Агрегации по осям

Сначала про форму: `reshape` меняет её, не трогая сами числа, — `np.arange(6).reshape(2, 3)` раскладывает те же шесть чисел по два ряда. Вместо одного из размеров можно написать `-1` («посчитай сам»): `reshape(-1, 1)` превращает вектор в столбец.

Дальше эти таблицы надо сворачивать: среднее по каждому признаку, максимум по каждому объекту — с этого начинается разбор любых данных. У `sum`, `mean`, `max` для этого есть параметр `axis` — вдоль какой оси «сворачивать». Правило простое: `axis=0` — сворачиваем строки, остаётся результат **по столбцам**; `axis=1` — наоборот. Без `axis` считается по всему массиву сразу.

axis — это та ось, которая пропадает

In [ ]:
M = np.arange(6).reshape(2, 3)   # 6 чисел укладываем в матрицу 2×3
M                                # последнее выражение печатается само, без print

Теперь свернём эту матрицу вдоль каждой оси и сравним формы результата:

In [ ]:
print("по столбцам (axis=0):", M.sum(axis=0))   # ось 0 схлопнулась: (2, 3) -> (3,)
print("по строкам  (axis=1):", M.sum(axis=1))   # ось 1 схлопнулась: (2, 3) -> (2,)
print("по всей матрице:     ", M.sum())         # без axis остаётся одно число
print("индекс максимума:    ", M.argmax())      # индекс в развёрнутом массиве: 5, а не (1, 2)

#### ❓ **Вопрос**: Почему `M.sum(axis=0)` для матрицы 2×3 даёт три числа, а не два?

<details>

<summary><strong>Ответ</strong></summary>

`axis=0` — это ось строк, и суммирование идёт **вдоль** неё: строки складываются друг с другом и исчезают, а ось столбцов остаётся. Форма `(2, 3)` превращается в `(3,)`. Полезно читать так: «`axis` — это та ось, которая пропадает».

</details>

### Broadcasting: массивы разной формы

Складывать поэлементно можно не только массивы одинаковой формы. **Broadcasting** — правило, по которому NumPy «растягивает» меньший массив до формы большего, не копируя данные в памяти.

Правило одно: формы сравниваются **справа налево**, и по каждой оси размеры должны либо совпадать, либо один из них должен быть равен 1 — такая ось и растягивается.

Broadcasting: формы сравниваются справа налево

Это не синтаксический сахар: именно так центрируют данные, нормируют признаки и считают попарные величины — без единого цикла Python.

In [ ]:
table = np.arange(12).reshape(3, 4).astype(float)
col_mean = table.mean(axis=0)            # среднее по столбцам, форма (4,)

print("формы:", table.shape, "и", col_mean.shape)
print(table - col_mean)                  # (3, 4) - (4,) -> вычлось из каждой строки

Ось длины 1 можно создать самому — индексом `None` (синоним `np.newaxis`). Так из двух одномерных массивов получают таблицу всех попарных сумм:

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20])

print((b[:, None] + a[None, :]).shape)   # (2, 1) + (1, 3) -> (2, 3)
print(b[:, None] + a[None, :])           # таблица всех попарных сумм b_i + a_j

Если формы несовместимы, NumPy не угадывает, а честно падает:

In [ ]:
try:
    np.arange(3) + np.arange(4)          # (3,) и (4,) — справа 3 против 4
except ValueError as error:
    print("ошибка:", error)

#### ❓ **Вопрос**: Почему `table - col_mean` сработало для форм `(3, 4)` и `(4,)`, а `np.arange(3) + np.arange(4)` — нет?

<details>

<summary><strong>Ответ</strong></summary>

Формы сравниваются справа налево. В первом случае: `4` против `4` — совпало, слева у второго массива оси просто нет, она достраивается как `1` и растягивается до `3`. Во втором случае самая правая ось — `3` против `4`: размеры не совпадают и ни один из них не равен 1, растягивать нечего, поэтому `ValueError`.

Практический вывод: когда broadcasting «не работает», сначала печатайте `.shape` обоих массивов — почти всегда ошибка именно в том, что ось оказалась не с той стороны. Развернуть её помогает индекс `None`.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Broadcasting — главный источник **тихих** багов у новичков, тех, что не падают. Классика жанра: предсказания модели имеют форму `(n, 1)`, а правильные ответы — `(n,)`. Разность `y_pred - y_true` не бросает исключение: по правилам она честно растягивает обе оси и выдаёт матрицу `(n, n)`. При `n = 100` вы просто получите бессмысленную метрику и полдня будете искать ошибку в модели; при `n = 100 000` — `MemoryError` на 80 гигабайт.

Отсюда рабочая привычка: перед арифметикой над двумя массивами печатать `.shape` обоих, а лишнюю ось убирать `.ravel()` или `.squeeze()`. И приятная деталь: растяжение ничего не копирует — NumPy ходит по одной и той же памяти с нулевым шагом, поэтому broadcasting не только короче цикла, но и дешевле.

</details>

## 4. Matplotlib: первый график

`matplotlib.pyplot` строит графики. Быстрый способ — вызвать `plt.plot(x, y)`, затем добавить подписи и показать. Для нескольких графиков на фигуре удобен объектный интерфейс: `fig, ax = plt.subplots()`.

Правила читаемого графика: подписать оси (`xlabel`, `ylabel`), дать заголовок (`title`), включить сетку (`grid`), а при нескольких кривых — легенду (`legend`; для неё у `plot` нужен `label=`). Сохранить фигуру в файл умеет `savefig` — и порядок вызовов здесь не произвольный, к этому вернёмся сразу после демки.

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 200)   # много точек → гладкая кривая
plt.plot(x, np.sin(x))
plt.xlabel("x")
plt.ylabel("sin(x)")
plt.show()

Теперь две кривые на одной фигуре — тут появляется легенда, а заодно сохраним результат в файл (он ляжет в `~/seminar-01/`, куда мы перешли в начале):

In [ ]:
plt.plot(x, np.sin(x), label="sin(x)")     # label нужен легенде
plt.plot(x, np.cos(x), label="cos(x)")
plt.title("sin и cos на [0, 2π]")
plt.legend()
plt.grid(True)
plt.savefig("sin_cos.png", dpi=150)        # СНАЧАЛА сохранить — файл ляжет в ~/seminar-01
plt.show()                                 # ...и только потом показать

In [ ]:
!ls -l sin_cos.png   # в выводе главное — размер файла: картинка не пустая

#### ❓ **Вопрос**: Почему `plt.savefig(...)` нужно вызывать до `plt.show()`?

<details>

<summary><strong>Ответ</strong></summary>

`plt.show()` отрисовывает и **очищает** текущую фигуру. Если вызвать `savefig` после `show()`, сохранять будет уже нечего — файл выйдет пустым. Поэтому порядок: сначала `savefig`, затем `show`.

</details>

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Matplotlib написал в 2003 году Джон Хантер — нейробиолог, который анализировал ЭКоГ пациентов с эпилепсией и хотел, чтобы лаборатория слезла с MATLAB. Отсюда и интерфейс `plt.*`: он сознательно повторял MATLAB с его «текущей фигурой», чтобы коллегам не пришлось переучиваться.

Это наследие мы и разбираем прямо сейчас: `plt.title` подписывает ту область, которую библиотека считает текущей, а `ax.set_title` — конкретную. Хантер умер в 2012-м, библиотеку с тех пор ведёт сообщество, а его именем NumFOCUS назвал стипендию для разработчиков научного софта.

</details>

### Несколько графиков на одной фигуре

Пока графиков один-два, хватает `plt.plot`. Для сетки графиков берут **объектный интерфейс**: `plt.subplots(rows, cols)` возвращает фигуру и массив областей (`axes`), у каждой — свои `plot`, `set_title`, `grid`. `tight_layout()` в конце разносит подписи, чтобы они не наезжали друг на друга.

Две детали, которые понадобятся в задачах: у сетки `2×2` массив областей **двумерный**, обращение к ним — `axes[0, 0]`, `axes[1, 1]`, а пройти по всем сразу удобно через `for ax in axes.flat`. И сохраняется такая фигура своим методом — `fig.savefig("subplots.png")`: он сохраняет именно эту фигуру, не полагаясь на то, какую matplotlib считает текущей.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))   # две области в ряд, axes — массив из двух

axes[0].plot(x, np.sin(x))                       # рисуем в левую область явно, без «текущей»
axes[0].set_title("sin(x)")                      # у области не title, а set_title
axes[1].plot(x, np.sin(x) ** 2)
axes[1].set_title("sin²(x)")
fig.tight_layout()                               # разносит подписи, чтобы не наезжали друг на друга
plt.show()

#### ❓ **Вопрос**: Чем `axes[0].set_title("...")` отличается от `plt.title("...")`?

<details>

<summary><strong>Ответ</strong></summary>

`plt.title` действует на «текущую» область — ту, которую matplotlib считает активной. Пока область одна, это удобно, но в сетке из четырёх графиков угадывать текущую опасно. `axes[0]` — прямая ссылка на конкретную область, и метод `set_title` подписывает именно её. Отсюда правило: один график — можно `plt.*`, несколько — только через `axes`.

</details>

### Разные типы графиков и оформление

Сегодня мы рисуем `plot`, `scatter` и `hist`, но типов графиков в matplotlib десятки (`bar`, `fill_between`, `imshow`, `contourf` — и дальше по [галерее](https://matplotlib.org/stable/gallery/index.html)). Устройство у всех одинаковое, меняется вызов; а вот оформление настраивается одними и теми же параметрами:

- **стиль линии** `linestyle`: `"-"` сплошная, `"--"` пунктир (dashed), `":"` точки (dotted), `"-."` штрихпунктир (dash-dot);
- **цвет** `color` (`"crimson"`, `"#1f77b4"`), **толщина** `linewidth`, **маркеры** `marker="o"`;
- **полупрозрачность** `alpha` — число от 0 (прозрачно) до 1 (непрозрачно).

In [ ]:
styles = [("-", "сплошная"), ("--", "пунктир"), (":", "точки"), ("-.", "штрихпунктир")]

plt.figure(figsize=(9, 4))
for i, (line_style, name) in enumerate(styles):
    plt.plot(x, np.sin(x - 0.6 * i), linestyle=line_style, linewidth=2, label=name)  # сдвиг по фазе, чтобы кривые не слиплись
plt.legend()
plt.grid(True, alpha=0.3)   # бледная сетка не спорит с данными
plt.show()

Другой тип графика — **диаграмма рассеяния** (`scatter`): она показывает не кривую, а облако наблюдений. Сначала сгенерируем данные, в которых есть зависимость и шум:

In [ ]:
rng = np.random.default_rng(0)                          # генератор с фиксированным seed: числа воспроизводимы
x_points = rng.normal(size=400)                         # 400 значений из нормального распределения
y_points = 0.6 * x_points + rng.normal(size=400) * 0.4  # линейная зависимость плюс шум
print("точек:", x_points.size)                          # данных много — они будут накладываться

Четыреста точек на одном рисунке неизбежно налезают друг на друга. Нарисуем их с `alpha=0.6` и посмотрим, что это даёт:

In [ ]:
plt.scatter(x_points, y_points, alpha=0.6,   # alpha < 1: перекрытия выходят темнее
            edgecolors="k", linewidths=0.3)  # тонкий контур отделяет соседние точки
plt.xlabel("x")
plt.ylabel("y")
plt.title("Диаграмма рассеяния: 400 точек, alpha=0.6")
plt.show()

### Гистограмма

`plt.hist` разбивает данные на интервалы (`bins`) и показывает, сколько значений попало в каждый — так видно форму распределения.

In [ ]:
sample = rng.normal(size=10_000)   # выборка из того же генератора, что и выше
plt.hist(sample, bins=50)          # 50 интервалов; по вертикали — количество значений
plt.title("Сколько значений попало в каждый интервал")
plt.show()

По вертикали сейчас **количество** наблюдений: удвоите выборку — удвоятся и столбики, поэтому сравнить такую гистограмму с формулой плотности нельзя. `density=True` меняет масштаб так, что **площадь** всех столбиков равна 1 — и теоретическую кривую можно нарисовать поверх, в тех же осях.

In [ ]:
grid = np.linspace(-4, 4, 200)                          # сетка для теоретической кривой
density = np.exp(-grid ** 2 / 2) / np.sqrt(2 * np.pi)   # плотность стандартного нормального N(0, 1)

plt.hist(sample, bins=50, density=True, alpha=0.6)      # density=True: площадь столбиков = 1
plt.plot(grid, density, color="crimson")                # кривая и гистограмма теперь в одном масштабе
plt.title("Гистограмма выборки и теоретическая плотность")
plt.show()

#### ❓ **Вопрос**: Зачем на диаграмме рассеяния полупрозрачность (`alpha`)?

<details>

<summary><strong>Ответ</strong></summary>

Когда точек много и они накладываются, при `alpha=1` всё сливается в сплошное пятно и не видно, где точек больше. Полупрозрачность делает перекрытия темнее, поэтому по насыщенности цвета читается плотность — где данных много, а где мало.

</details>

## 5. Интерактивные графики: plotly

Matplotlib рисует **статичную** картинку: что нарисовано, то и видно. Когда точек много и хочется разглядеть отдельные — нужен **интерактив**: зум, панорама, подсказка по наведению. Это умеет `plotly`: график остаётся живым прямо в ячейке ноутбука, в Colab — из коробки. Практическое правило: отчёт и статья — matplotlib, разведка данных и демонстрация коллеге — plotly.

Установка (если пакета нет): `!pip install plotly`.

Данные `plotly` берёт из таблиц **pandas** — это библиотека для табличных данных, ей посвящён семинар 13. Сегодня достаточно смотреть на `df` как на таблицу с именованными колонками, а `df.head()` — как на «покажи первые пять строк».

In [ ]:
!pip install -q plotly       # в Colab plotly уже есть, в свежем venv — нет
import plotly.express as px

df = px.data.iris()          # встроенный набор данных: таблица pandas с колонками
df.head()                    # первые пять строк — что вообще лежит в данных

Что лежит в данных — посмотрели. Теперь сам график: цвет кодирует вид ириса, размер точки — длину лепестка.

In [ ]:
fig = px.scatter(df, x="sepal_width", y="sepal_length",
                 color="species", size="petal_length",   # цвет и размер точки — тоже данные
                 title="Ирисы Фишера — наведи курсор на точку")
fig.show()                       # график интерактивный: зум колесом, подсказка по наведению
fig.write_html("iris.html")      # тот же интерактив файлом — можно отправить коллеге

#### ❓ **Вопрос**: Тот же набор точек можно нарисовать и в matplotlib. Что даёт здесь plotly, чего не даёт статичная картинка?

<details>

<summary><strong>Ответ</strong></summary>

На статичной картинке точка — это только положение, цвет и размер: увидеть, **что** это за наблюдение, нельзя. В plotly-графике выше при наведении всплывает подсказка с конкретными значениями (`sepal_width`, `sepal_length`, `species`, `petal_length`), а плотное облако можно приблизить зумом и разглядеть отдельные точки. Плата за это — график живёт только в браузере: в PDF или в печатный отчёт он не переносится, туда идёт matplotlib.

</details>

## 6. Google Colab

Открыть этот ноутбук в Colab: [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** (или **Open notebook → GitHub** и ссылка на файл в репозитории курса).

<!-- TODO: когда адрес итогового репозитория курса зафиксируется, поставить здесь бейдж «Открыть в Colab»
     со ссылкой вида https://colab.research.google.com/github/<org>/<repo>/blob/main/seminars/01-jupyter-viz/demo.ipynb -->

**Google Colab** — это Jupyter в облаке: ноутбук исполняется на удалённой машине Google с бесплатным доступом к GPU/TPU (Runtime → Change runtime type). Основные библиотеки (`numpy`, `matplotlib`, `pandas`, `torch`) уже предустановлены; недостающее ставят через `!pip install ...`. Проверить GPU — `!nvidia-smi`.

**Сессия одноразовая.** Машина выдаётся на несколько часов и отключается при простое, а вместе с ней исчезает всё, что вы в ней сделали: и доустановленные пакеты, и файлы в рабочем каталоге. Отсюда два следствия — строку `!pip install ...` держат прямо в первой ячейке ноутбука, а результаты сохраняют наружу (как именно — вопрос ниже).

Бесплатные GPU при этом не гарантированы и выдаются по остаточному принципу: для долгих расчётов это неудобно, для семинаров и небольших экспериментов — вполне достаточно.

<details>

<summary>🎙 <em>Заметка преподавателя — рассказывается вслух</em></summary>

Colab появился в конце 2017-го и быстро стал стандартным способом раздать код: даёшь ссылку — и у человека уже есть машина с GPU, ставить ничего не надо. Обратная сторона — сотни ноутбуков в интернете написаны ровно под Colab: первой строкой `!pip install ...`, пути ведут в `/content`, версии библиотек те, что были в Colab на момент написания.

Когда такой ноутбук впервые запускают на своей машине, он ломается на первой же ячейке — и это не потому, что «код плохой», а потому, что окружение никто не зафиксировал. Как собирают своё окружение и почему это отдельная работа — семинар 5.

</details>

#### ❓ **Вопрос**: Куда сохранять результаты в Colab, чтобы они не пропали после отключения среды?

<details>

<summary><strong>Ответ</strong></summary>

Наружу — рабочий каталог сессии живёт ровно столько же, сколько сама машина. Обычный способ — примонтировать Google Drive и писать в него:

```python
from google.colab import drive
drive.mount('/content/drive')      # файлы появятся в /content/drive/MyDrive
```

Дальше `plt.savefig("/content/drive/MyDrive/plot.png")` — и картинка переживёт закрытие ноутбука. Разовый файл можно просто скачать себе: `files.download("plot.png")` из модуля `google.colab`.

</details>